# Real-Market Implied Volatility — Cleaned Version

This notebook calculates the implied volatility of a live SPY option from Yahoo Finance market data and compares it with Yahoo's reported `impliedVolatility`.

### Improvements over the original notebook
- Rejects zero/invalid bid-ask quotes instead of falling back to them.
- Uses the bid-ask midpoint as the market option price.
- Uses the actual option expiration time (4:00 PM US/Eastern) for time-to-expiry.
- Includes an optional continuous dividend yield.
- Checks no-arbitrage price bounds before solving.
- Calculates IV with both **Brent's method** and **Newton-Raphson**.
- Verifies the calculated IV by repricing the option.
- Reports the difference from Yahoo's IV.
- Keeps the model assumptions explicit, so differences from Yahoo can be diagnosed rather than hidden.

> Note: Yahoo's IV is a vendor-provided value. An exact match is not guaranteed because Yahoo may use different assumptions for rates, dividends, timing, and option conventions.

In [1]:
# Install if required:
# %pip install yfinance scipy pandas numpy matplotlib

import math
import time
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import brentq
from scipy.stats import norm

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

EASTERN = ZoneInfo("America/New_York")


## 1. Black-Scholes-Merton pricing functions

In [2]:
def _d1(S, K, T, r, q, sigma):
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        raise ValueError("S, K, T and sigma must be positive.")
    return (
        math.log(S / K)
        + (r - q + 0.5 * sigma**2) * T
    ) / (sigma * math.sqrt(T))


def _d2(S, K, T, r, q, sigma):
    return _d1(S, K, T, r, q, sigma) - sigma * math.sqrt(T)


def black_scholes_price(S, K, T, r, q, sigma, option_type="call"):
    """Black-Scholes-Merton price for a European call or put."""
    if T <= 0:
        if option_type.lower() == "call":
            return max(S - K, 0.0)
        return max(K - S, 0.0)

    d1 = _d1(S, K, T, r, q, sigma)
    d2 = d1 - sigma * math.sqrt(T)

    if option_type.lower() == "call":
        return (
            S * math.exp(-q * T) * norm.cdf(d1)
            - K * math.exp(-r * T) * norm.cdf(d2)
        )
    else:
        return (
            K * math.exp(-r * T) * norm.cdf(-d2)
            - S * math.exp(-q * T) * norm.cdf(-d1)
        )


def black_scholes_vega(S, K, T, r, q, sigma):
    """Vega = dPrice/dSigma, with sigma expressed as a decimal."""
    if T <= 0 or sigma <= 0:
        return 0.0

    d1 = _d1(S, K, T, r, q, sigma)
    return S * math.exp(-q * T) * norm.pdf(d1) * math.sqrt(T)


def black_scholes_greeks(S, K, T, r, q, sigma, option_type="call"):
    """Return common BSM Greeks for the selected option."""
    d1 = _d1(S, K, T, r, q, sigma)
    d2 = d1 - sigma * math.sqrt(T)

    delta = (
        math.exp(-q * T) * norm.cdf(d1)
        if option_type.lower() == "call"
        else math.exp(-q * T) * (norm.cdf(d1) - 1)
    )

    gamma = (
        math.exp(-q * T) * norm.pdf(d1)
        / (S * sigma * math.sqrt(T))
    )

    vega = black_scholes_vega(S, K, T, r, q, sigma)

    if option_type.lower() == "call":
        theta = (
            -S * math.exp(-q * T) * norm.pdf(d1) * sigma / (2 * math.sqrt(T))
            - r * K * math.exp(-r * T) * norm.cdf(d2)
            + q * S * math.exp(-q * T) * norm.cdf(d1)
        )
    else:
        theta = (
            -S * math.exp(-q * T) * norm.pdf(d1) * sigma / (2 * math.sqrt(T))
            + r * K * math.exp(-r * T) * norm.cdf(-d2)
            - q * S * math.exp(-q * T) * norm.cdf(-d1)
        )

    return {
        "delta": delta,
        "gamma": gamma,
        "vega": vega,
        "theta_per_year": theta,
        "theta_per_day": theta / 365.0,
    }


## 2. Implied-volatility solvers

In [3]:
def no_arbitrage_bounds(S, K, T, r, q, option_type="call"):
    """European option price bounds under BSM assumptions."""
    if option_type.lower() == "call":
        lower = max(S * math.exp(-q * T) - K * math.exp(-r * T), 0.0)
        upper = S * math.exp(-q * T)
    else:
        lower = max(K * math.exp(-r * T) - S * math.exp(-q * T), 0.0)
        upper = K * math.exp(-r * T)

    return lower, upper


def implied_vol_brent(
    market_price,
    S,
    K,
    T,
    r,
    q=0.0,
    option_type="call",
    lower_sigma=1e-6,
    upper_sigma=5.0,
):
    """Robust IV calculation using Brent's root-finding method."""

    if not np.isfinite(market_price):
        return np.nan

    lower_bound, upper_bound = no_arbitrage_bounds(
        S, K, T, r, q, option_type
    )

    # Small tolerance prevents rejecting a valid quote because of floating-point noise.
    tolerance = 1e-10

    if market_price < lower_bound - tolerance:
        return np.nan

    if market_price > upper_bound + tolerance:
        return np.nan

    def objective(sigma):
        return (
            black_scholes_price(S, K, T, r, q, sigma, option_type)
            - market_price
        )

    try:
        return brentq(
            objective,
            a=lower_sigma,
            b=upper_sigma,
            xtol=1e-12,
            rtol=1e-12,
            maxiter=200,
        )
    except (ValueError, RuntimeError):
        return np.nan


def implied_vol_newton(
    market_price,
    S,
    K,
    T,
    r,
    q=0.0,
    option_type="call",
    initial_guess=0.20,
    tolerance=1e-10,
    max_iter=100,
):
    """Newton-Raphson IV calculation using analytical BSM vega."""

    if not np.isfinite(market_price):
        return np.nan

    lower_bound, upper_bound = no_arbitrage_bounds(
        S, K, T, r, q, option_type
    )

    if market_price < lower_bound - 1e-10 or market_price > upper_bound + 1e-10:
        return np.nan

    sigma = max(float(initial_guess), 1e-6)

    for _ in range(max_iter):
        price = black_scholes_price(
            S, K, T, r, q, sigma, option_type
        )
        vega = black_scholes_vega(S, K, T, r, q, sigma)

        price_error = price - market_price

        if abs(price_error) < tolerance:
            return sigma

        if abs(vega) < 1e-12:
            return np.nan

        sigma_new = sigma - price_error / vega

        # Keep Newton inside a sensible positive volatility range.
        if not np.isfinite(sigma_new) or sigma_new <= 0 or sigma_new > 10:
            return np.nan

        sigma = sigma_new

    return np.nan


## 3. Time-to-expiry and market-data helpers

In [4]:
def expiration_datetime(expiration_date):
    """Yahoo supplies the expiration date; use 4:00 PM US/Eastern."""
    expiry = datetime.strptime(expiration_date, "%Y-%m-%d")
    return expiry.replace(
        hour=16,
        minute=0,
        second=0,
        microsecond=0,
        tzinfo=EASTERN,
    )


def calculate_time_to_expiry(expiration_date):
    """Actual time remaining in years using 365-day annualization."""
    now = datetime.now(EASTERN)
    expiry = expiration_datetime(expiration_date)

    seconds = (expiry - now).total_seconds()
    return max(seconds / (365.0 * 24.0 * 60.0 * 60.0), 1e-10)


def get_spot_price(ticker):
    """Get the latest underlying price with a small fallback chain."""
    try:
        spot = ticker.fast_info["lastPrice"]
        if spot is not None and np.isfinite(float(spot)):
            return float(spot)
    except Exception:
        pass

    hist = ticker.history(period="1d")
    if hist.empty:
        raise ValueError("Could not obtain the underlying spot price.")

    return float(hist["Close"].dropna().iloc[-1])


def get_risk_free_rate():
    """Use 13-week T-bill (^IRX) as a transparent short-rate proxy."""
    irx = yf.Ticker("^IRX")

    try:
        value = irx.fast_info["lastPrice"]
        if value is not None and np.isfinite(float(value)):
            return float(value) / 100.0
    except Exception:
        pass

    hist = irx.history(period="5d")
    if hist.empty:
        raise ValueError("Could not obtain ^IRX risk-free-rate proxy.")

    return float(hist["Close"].dropna().iloc[-1]) / 100.0


def get_dividend_yield(ticker):
    """Get Yahoo's current dividend-yield field when available.

    This is an approximation to continuous dividend yield, not a guarantee
    of the exact dividend/carry convention used by Yahoo's IV calculation.
    """
    try:
        info = ticker.info
        q = info.get("dividendYield")

        if q is not None and np.isfinite(float(q)):
            return max(float(q), 0.0)
    except Exception:
        pass

    return 0.0


## 4. Download live SPY data

In [5]:
ticker_symbol = "SPY"

print(f"Fetching live data for {ticker_symbol}...")
ticker = yf.Ticker(ticker_symbol)

spot_price = get_spot_price(ticker)
risk_free_rate = get_risk_free_rate()
dividend_yield = get_dividend_yield(ticker)

print(f"Spot price:       ${spot_price:.4f}")
print(f"Risk-free proxy:  {risk_free_rate * 100:.4f}%")
print(f"Dividend yield:   {dividend_yield * 100:.4f}%")

expirations = ticker.options

if not expirations:
    raise ValueError("No option expiration dates returned by Yahoo Finance.")

print(f"Number of expirations available: {len(expirations)}")
print(f"First 10 expirations: {expirations[:10]}")


Fetching live data for SPY...
Spot price:       $765.7200
Risk-free proxy:  3.7100%
Dividend yield:   101.0000%
Number of expirations available: 29
First 10 expirations: ('2026-08-24', '2026-08-25', '2026-08-26', '2026-08-27', '2026-08-28', '2026-08-31', '2026-09-04', '2026-09-11', '2026-09-18', '2026-09-25')


## 5. Select an expiration near 30 days

In [6]:
today = datetime.now(EASTERN)

target_expiry = None
best_distance = float("inf")

for exp in expirations:
    expiry_dt = expiration_datetime(exp)
    days_out = (expiry_dt - today).total_seconds() / (24 * 3600)

    # Ignore already-expired dates.
    if days_out <= 0:
        continue

    distance = abs(days_out - 30)

    if distance < best_distance:
        best_distance = distance
        target_expiry = exp

if target_expiry is None:
    raise ValueError("No future expiration date was found.")

T = calculate_time_to_expiry(target_expiry)

print(f"Selected expiry: {target_expiry}")
print(f"Time to expiry:  {T:.6f} years")
print(f"Approx. days:    {T * 365:.3f}")


Selected expiry: 2026-09-25
Time to expiry:  0.091654 years
Approx. days:    33.454


## 6. Download and clean the option chain

In [7]:
chain = ticker.option_chain(target_expiry)
calls = chain.calls.copy()

# Keep only quotes that can form a meaningful bid-ask midpoint.
liquid_calls = calls[
    (calls["bid"] > 0) &
    (calls["ask"] > 0) &
    (calls["ask"] >= calls["bid"]) &
    calls["strike"].notna() &
    calls["impliedVolatility"].notna() &
    (calls["impliedVolatility"] > 0)
].copy()

if liquid_calls.empty:
    raise ValueError(
        "No valid liquid call quotes were returned for the selected expiry."
    )

# Midpoint is the market price used for our IV calculation.
liquid_calls["mid"] = (
    liquid_calls["bid"] + liquid_calls["ask"]
) / 2.0

# Select the strike closest to spot.
liquid_calls["strike_distance"] = (
    liquid_calls["strike"] - spot_price
).abs()

atm_call = liquid_calls.loc[
    liquid_calls["strike_distance"].idxmin()
]

strike = float(atm_call["strike"])
bid = float(atm_call["bid"])
ask = float(atm_call["ask"])
market_price = float(atm_call["mid"])
yahoo_iv = float(atm_call["impliedVolatility"])

print(f"Valid liquid calls: {len(liquid_calls)}")
print()
print("[Selected ATM Call]")
print(f"Strike:       ${strike:.4f}")
print(f"Bid:          ${bid:.4f}")
print(f"Ask:          ${ask:.4f}")
print(f"Midpoint:     ${market_price:.4f}")
print(f"Yahoo IV:     {yahoo_iv * 100:.4f}%")


Valid liquid calls: 120

[Selected ATM Call]
Strike:       $766.0000
Bid:          $12.2000
Ask:          $12.2500
Midpoint:     $12.2250
Yahoo IV:     13.2836%


## 7. Validate the market price before solving for IV

In [8]:
lower_bound, upper_bound = no_arbitrage_bounds(
    spot_price,
    strike,
    T,
    risk_free_rate,
    dividend_yield,
    "call",
)

print("[No-Arbitrage Check]")
print(f"European BSM lower bound: ${lower_bound:.6f}")
print(f"European BSM upper bound: ${upper_bound:.6f}")
print(f"Observed midpoint:        ${market_price:.6f}")

if not (lower_bound <= market_price <= upper_bound):
    raise ValueError(
        "The selected market midpoint violates the BSM price bounds. "
        "Do not calculate IV from this quote."
    )

print("Result: market price is within the BSM bounds.")


[No-Arbitrage Check]
European BSM lower bound: $0.000000
European BSM upper bound: $698.018427
Observed midpoint:        $12.225000
Result: market price is within the BSM bounds.


## 8. Calculate IV using Brent and Newton-Raphson

In [9]:
start = time.perf_counter()

iv_brent = implied_vol_brent(
    market_price=market_price,
    S=spot_price,
    K=strike,
    T=T,
    r=risk_free_rate,
    q=dividend_yield,
    option_type="call",
)

brent_time_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()

iv_newton = implied_vol_newton(
    market_price=market_price,
    S=spot_price,
    K=strike,
    T=T,
    r=risk_free_rate,
    q=dividend_yield,
    option_type="call",
    initial_guess=0.20,
)

newton_time_ms = (time.perf_counter() - start) * 1000

print("[Implied Volatility Results]")
print(f"Brent IV:       {iv_brent * 100:.6f}%")
print(f"Newton IV:      {iv_newton * 100:.6f}%")
print(f"Yahoo IV:       {yahoo_iv * 100:.6f}%")
print()
print(f"Brent time:     {brent_time_ms:.4f} ms")
print(f"Newton time:    {newton_time_ms:.4f} ms")
print()
print(f"Brent vs Yahoo:  {abs(iv_brent - yahoo_iv) * 100:.6f} percentage points")
print(f"Newton vs Yahoo: {abs(iv_newton - yahoo_iv) * 100:.6f} percentage points")


[Implied Volatility Results]
Brent IV:       40.636500%
Newton IV:      40.636500%
Yahoo IV:       13.283643%

Brent time:     21.5266 ms
Newton time:    4.7910 ms

Brent vs Yahoo:  27.352857 percentage points
Newton vs Yahoo: 27.352857 percentage points


## 9. Verify the calculated IV by repricing the option

In [10]:
calculated_price_brent = black_scholes_price(
    spot_price,
    strike,
    T,
    risk_free_rate,
    dividend_yield,
    iv_brent,
    "call",
)

calculated_price_newton = black_scholes_price(
    spot_price,
    strike,
    T,
    risk_free_rate,
    dividend_yield,
    iv_newton,
    "call",
)

print("[Price Verification]")
print(f"Observed midpoint:       ${market_price:.8f}")
print(f"Price from Brent IV:     ${calculated_price_brent:.8f}")
print(f"Price from Newton IV:    ${calculated_price_newton:.8f}")
print()
print(f"Brent pricing error:     ${abs(calculated_price_brent - market_price):.10f}")
print(f"Newton pricing error:    ${abs(calculated_price_newton - market_price):.10f}")


[Price Verification]
Observed midpoint:       $12.22500000
Price from Brent IV:     $12.22500000
Price from Newton IV:    $12.22500000

Brent pricing error:     $0.0000000000
Newton pricing error:    $0.0000000000


## 10. Calculate Greeks using the calculated IV

In [ ]:
greeks = black_scholes_greeks(
    S=spot_price,
    K=strike,
    T=T,
    r=risk_free_rate,
    q=dividend_yield,
    sigma=iv_brent,
    option_type="call",
)

print("[Live-Market Greeks]")
print(f"Delta:          {greeks['delta']:.6f}")
print(f"Gamma:          {greeks['gamma']:.6f}")
print(f"Vega:           {greeks['vega']:.6f}")
print(f"Theta / year:   {greeks['theta_per_year']:.6f}")
print(f"Theta / day:    {greeks['theta_per_day']:.6f}")


## 11. Final comparison table

In [11]:
comparison = pd.DataFrame({
    "Metric": [
        "Spot",
        "Strike",
        "Time to expiry (years)",
        "Bid",
        "Ask",
        "Midpoint market price",
        "Risk-free rate",
        "Dividend yield",
        "Yahoo IV",
        "Brent IV",
        "Newton IV",
        "Brent - Yahoo",
        "Newton - Yahoo",
        "Brent repricing error",
        "Newton repricing error",
    ],
    "Value": [
        spot_price,
        strike,
        T,
        bid,
        ask,
        market_price,
        risk_free_rate,
        dividend_yield,
        yahoo_iv,
        iv_brent,
        iv_newton,
        iv_brent - yahoo_iv,
        iv_newton - yahoo_iv,
        calculated_price_brent - market_price,
        calculated_price_newton - market_price,
    ]
})

comparison


,Metric,Value
0,Spot,7.657200e+02
1,Strike,7.660000e+02
2,Time to expiry (years),9.165448e-02
3,Bid,1.220000e+01
4,Ask,1.225000e+01
5,Midpoint market price,1.222500e+01
6,Risk-free rate,3.710000e-02
7,Dividend yield,1.010000e+00
8,Yahoo IV,1.328364e-01
9,Brent IV,4.063650e-01


## Interpretation

### What should happen
If the quote is valid, both our Brent and Newton-Raphson calculations should recover essentially the same IV because they solve the same BSM equation:

\[
BS(S,K,T,r,q,\sigma)=P_{mid}
\]

The repricing error should be extremely small.

### Why our IV may still differ from Yahoo
A remaining difference does **not automatically indicate a coding error**. Possible sources include:

1. Yahoo and this notebook may use different risk-free-rate assumptions.
2. The dividend yield used here is an approximation to continuous dividend yield.
3. Yahoo may use a different time-to-expiry convention.
4. SPY options are American-style while the notebook uses the European BSM model.
5. The midpoint is our chosen market price; Yahoo may use a different option-price input/convention.
6. Market quotes can change between the underlying price and option-chain requests.

The important diagnostic is therefore:

**market quote → BSM IV → reprice → market quote**

If that loop closes accurately, the IV solver is working correctly.

In [12]:
yahoo_price = black_scholes_price(
    S=spot_price,
    K=strike,
    T=T,
    r=risk_free_rate,
    q=dividend_yield,
    sigma=yahoo_iv,
    option_type="call"
)

print(f"Market midpoint: ${market_price:.4f}")
print(f"Price using Yahoo IV: ${yahoo_price:.4f}")
print(f"Difference: ${yahoo_price - market_price:.4f}")

Market midpoint: $12.2250
Price using Yahoo IV: $0.1330
Difference: $-12.0920
